In [1]:
import pickle
from pathlib import Path
import numpy as np
import scipy as sp
import os
from collections import defaultdict

import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import random

from tqdm import tqdm

gpu = "1"
device = torch.device(f"cuda:{gpu}" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
batch_size = 128
dropout_mlp = 0.5
dropout_gru = 0.25
learning_rate = 1e-4
weight_decay = 1e-2

Using device: cuda:1


In [2]:
inference_results = list(Path("./results/10000_samples_OpenLlama").rglob("*.pickle"))
print(inference_results)
# os.makedirs("./models/", exist_ok=True)

[PosixPath('results/10000_samples_OpenLlama/open_llama_7b_trivia_qa_start-0_end-10000_6_16.pickle')]


### Single layer classifier - Originally from the paper

In [3]:
class FFHallucinationClassifier(torch.nn.Module):
    def __init__(self, input_shape, dropout = dropout_mlp):
        super().__init__()
        self.dropout = dropout
        
        self.linear_relu_stack =torch.nn.Sequential(
            torch.nn.Linear(input_shape, 256),
            torch.nn.ReLU(),
            torch.nn.Dropout(self.dropout),
            torch.nn.Linear(256, 2)
            )

    def forward(self, x):
        logits = self.linear_relu_stack(x)
        return logits

class RNNHallucinationClassifier(torch.nn.Module):
    def __init__(self, dropout=dropout_gru):
        super().__init__()
        hidden_dim = 128
        num_layers = 4
        self.gru = torch.nn.GRU(1, hidden_dim, num_layers, dropout=dropout, batch_first=True, bidirectional=False)
        self.linear = torch.nn.Linear(hidden_dim, 2)
    
    def forward(self, seq):
        gru_out, _ = self.gru(seq)
        return self.linear(gru_out)[-1, -1, :]

In [4]:
def gen_classifier_roc(inputs, save_name="mlp_model.pt"):
    X_train, X_test, y_train, y_test = train_test_split(inputs, correct.astype(int), test_size=0.2, random_state=123)
    classifier_model = FFHallucinationClassifier(X_train.shape[1]).to(device)
    X_train = torch.tensor(X_train).to(device)
    y_train = torch.tensor(y_train).to(torch.long).to(device)
    X_test = torch.tensor(X_test).to(device)
    y_test = torch.tensor(y_test).to(torch.long).to(device)

    optimizer = torch.optim.AdamW(classifier_model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    for _ in tqdm(range(1001)):
        optimizer.zero_grad()
        sample = torch.randperm(X_train.shape[0])[:batch_size]
        pred = classifier_model(X_train[sample])
        loss = torch.nn.functional.cross_entropy(pred, y_train[sample])
        loss.backward()
        optimizer.step()
    # torch.save(classifier_model.state_dict(), f"./models/{save_name}")
    classifier_model.eval()
    with torch.no_grad():
        pred = torch.nn.functional.softmax(classifier_model(X_test), dim=1)
        prediction_classes = (pred[:,1]>0.5).type(torch.long).cpu()
        return roc_auc_score(y_test.cpu(), pred[:,1].cpu()), (prediction_classes.numpy()==y_test.cpu().numpy()).mean()

In [5]:
all_results = {}

In [ ]:
for results_file in inference_results:
    if results_file not in all_results.keys():
        try:
            del results
        except:
            pass
        try:
            classifier_results = {}
            with open(results_file, "rb") as infile:
                results = pickle.loads(infile.read())
            correct = np.array(results['correct'])
            infile.close()
    
            # attributes
            X_train, X_test, y_train, y_test = train_test_split(results['attributes_first'], correct.astype(int), test_size=0.2, random_state=1234)
            rnn_model = RNNHallucinationClassifier().to(device)
            optimizer = torch.optim.AdamW(rnn_model.parameters(), lr=learning_rate, weight_decay=weight_decay)
            for step in tqdm(range(1001)):
                x_sub, y_sub = zip(*random.sample(list(zip(X_train, y_train)), batch_size))
                y_sub = torch.tensor(y_sub).to(torch.long).to(device)
                optimizer.zero_grad()
                preds = torch.stack([rnn_model(torch.tensor(i).view(1, -1, 1).to(torch.float).to(device)) for i in x_sub])
                loss = torch.nn.functional.cross_entropy(preds, y_sub)
                loss.backward()
                optimizer.step()
            # torch.save(rnn_model.state_dict(), f"rnn_model.pt")
            
            preds = torch.stack([rnn_model(torch.tensor(i).view(1, -1, 1).to(torch.float).to(device)) for i in X_test])
            preds = torch.nn.functional.softmax(preds, dim=1)
            prediction_classes = (preds[:,1]>0.5).type(torch.long).cpu()
            classifier_results['attribution_rnn_roc'] = roc_auc_score(y_test, preds[:,1].detach().cpu().numpy())
            classifier_results['attribution_rnn_acc'] = (prediction_classes.numpy()==y_test).mean()

            # logits
            first_logits = np.stack([sp.special.softmax(i[j]) for i, j in zip(results['logits'], results['start_pos'])])
            first_logits_roc, first_logits_acc = gen_classifier_roc(first_logits, save_name="mlp_logit.pt")
            classifier_results['first_logits_roc'] = first_logits_roc
            classifier_results['first_logits_acc'] = first_logits_acc

            # fully connected
            for layer in range(results['first_fully_connected'][0].shape[0]):
                layer_roc, layer_acc = gen_classifier_roc(np.stack([i[layer] for i in results['first_fully_connected']]), save_name=f"mlp_fc_{layer}.pt")
                classifier_results[f'first_fully_connected_roc_{layer}'] = layer_roc
                classifier_results[f'first_fully_connected_acc_{layer}'] = layer_acc

            # attention
            for layer in range(results['first_attention'][0].shape[0]):
                layer_roc, layer_acc = gen_classifier_roc(np.stack([i[layer] for i in results['first_attention']]), save_name=f"mlp_attn_{layer}.pt")
                classifier_results[f'first_attention_roc_{layer}'] = layer_roc
                classifier_results[f'first_attention_acc_{layer}'] = layer_acc
            
            all_results[results_file] = classifier_results.copy()
        except Exception as err:
            print(err)

In [12]:
print(all_results.keys())

dict_keys([PosixPath('results/10000_samples/open_llama_7b_trivia_qa_start-0_end-10000_6_9.pickle')])


In [15]:
for k,v in all_results.items():
    for kk, vv in v.items():
        print(kk, vv)

attribution_rnn_roc 0.6205691910290967
attribution_rnn_acc 0.6395
first_logits_roc 0.6961727884506936
first_logits_acc 0.61
first_fully_connected_roc_0 0.6692200208027871
first_fully_connected_acc_0 0.6555
first_fully_connected_roc_1 0.6718402112025933
first_fully_connected_acc_1 0.6575
first_fully_connected_roc_2 0.6812495039496054
first_fully_connected_acc_2 0.655
first_fully_connected_roc_3 0.6978029666946544
first_fully_connected_acc_3 0.65
first_fully_connected_roc_4 0.7023791621238894
first_fully_connected_acc_4 0.676
first_fully_connected_roc_5 0.7085385415491811
first_fully_connected_acc_5 0.6825
first_fully_connected_roc_6 0.7123586517454709
first_fully_connected_acc_6 0.6905
first_fully_connected_roc_7 0.7156357172993137
first_fully_connected_acc_7 0.688
first_fully_connected_roc_8 0.7171364002823832
first_fully_connected_acc_8 0.6905
first_fully_connected_roc_9 0.7173849476379647
first_fully_connected_acc_9 0.686
first_fully_connected_roc_10 0.7202589069764527
first_fully_co

In [2]:
import pickle
import glob
from collections import defaultdict
from tqdm import tqdm
import os

results_dir = "./results/10000_samples_Qwen2.5"
pattern = os.path.join(results_dir, "*_batch_*.pickle")

batch_files = sorted(glob.glob(pattern))
if not batch_files:
    raise FileNotFoundError("No batch pickle files found.")

print(f"Found {len(batch_files)} batch files.")

# Initialize empty lists
combined = defaultdict(list)

# Iterate over each batch file and extend lists
for bf in tqdm(batch_files, desc="Merging batches"):
    with open(bf, "rb") as f:
        batch_data = pickle.load(f)
    for key, values in batch_data.items():
        combined[key].extend(values)
    f.close()
        
output_file = os.path.join(results_dir, "Qwen2.5-7B-Instruct_trivia_qa_start-0_end-10000.pickle")
with open(output_file, "wb") as f:
    pickle.dump(dict(combined), f)

print(f"Combined results saved to {output_file}")

### Eval on TRex

In [ ]:
def eval_mlp_logit(inputs):
    _, X_test, _, y_test = train_test_split(inputs, correct.astype(int), test_size=0.9, random_state=123)
    classifier_model = FFHallucinationClassifier(X_test.shape[1]).to(device)
    classifier_model.load_state_dict(torch.load("models/mlp_logit.pt", map_location=device))
    X_test = torch.tensor(X_test).to(device)
    y_test = torch.tensor(y_test).to(torch.long).to(device)
    classifier_model.eval()
    with torch.no_grad():
        pred = torch.nn.functional.softmax(classifier_model(X_test), dim=1)
        prediction_classes = (pred[:,1]>0.5).type(torch.long).cpu()
        return roc_auc_score(y_test.cpu(), pred[:,1].cpu()), (prediction_classes.numpy()==y_test.cpu().numpy()).mean()
    
def eval_mlp_fc(inputs, layer):
    _, X_test, _, y_test = train_test_split(inputs, correct.astype(int), test_size=0.9, random_state=123)
    classifier_model = FFHallucinationClassifier(X_test.shape[1]).to(device)
    classifier_model.load_state_dict(torch.load(f"models/mlp_fc_{layer}.pt", map_location=device))
    X_test = torch.tensor(X_test).to(device)
    y_test = torch.tensor(y_test).to(torch.long).to(device)
    classifier_model.eval()
    with torch.no_grad():
        pred = torch.nn.functional.softmax(classifier_model(X_test), dim=1)
        prediction_classes = (pred[:,1]>0.5).type(torch.long).cpu()
        return roc_auc_score(y_test.cpu(), pred[:,1].cpu()), (prediction_classes.numpy()==y_test.cpu().numpy()).mean()
    
def eval_mlp_attn(inputs, layer):
    _, X_test, _, y_test = train_test_split(inputs, correct.astype(int), test_size=0.9, random_state=123)
    classifier_model = FFHallucinationClassifier(X_test.shape[1]).to(device)
    classifier_model.load_state_dict(torch.load(f"models/mlp_attn_{layer}.pt", map_location=device))
    X_test = torch.tensor(X_test).to(device)
    y_test = torch.tensor(y_test).to(torch.long).to(device)
    classifier_model.eval()
    with torch.no_grad():
        pred = torch.nn.functional.softmax(classifier_model(X_test), dim=1)
        prediction_classes = (pred[:,1]>0.5).type(torch.long).cpu()
        return roc_auc_score(y_test.cpu(), pred[:,1].cpu()), (prediction_classes.numpy()==y_test.cpu().numpy()).mean()

In [ ]:
inference_results = list(Path("./results/trex").rglob("*.pickle"))
print(inference_results)
for results_file in inference_results:
    if results_file not in all_results.keys():
        try:
            del results
        except:
            pass
        try:
            classifier_results = {}
            with open(results_file, "rb") as infile:
                results = pickle.loads(infile.read())
            correct = np.array(results['correct'])
    
            # attributes
            X_train, X_test, y_train, y_test = train_test_split(results['attributes_first'], correct.astype(int), test_size=0.9, random_state=1234)
            rnn_model = RNNHallucinationClassifier().to(device)
            rnn_model.load_state_dict(torch.load("models/rnn_model.pt", map_location=device))
            rnn_model.eval()
            # optimizer = torch.optim.AdamW(rnn_model.parameters(), lr=learning_rate, weight_decay=weight_decay)
            # for step in tqdm(range(1001)):
            #     x_sub, y_sub = zip(*random.sample(list(zip(X_train, y_train)), batch_size))
            #     y_sub = torch.tensor(y_sub).to(torch.long)
            #     optimizer.zero_grad()
            #     preds = torch.stack([rnn_model(torch.tensor(i).view(1, -1, 1).to(torch.float)) for i in x_sub])
            #     loss = torch.nn.functional.cross_entropy(preds, y_sub)
            #     loss.backward()
            #     optimizer.step()
            # torch.save(rnn_model.state_dict(), f"rnn_model.pt")
            
            preds = torch.stack([rnn_model(torch.tensor(i).view(1, -1, 1).to(torch.float)) for i in X_test])
            preds = torch.nn.functional.softmax(preds, dim=1)
            prediction_classes = (preds[:,1]>0.5).type(torch.long).cpu()
            classifier_results['attribution_rnn_roc'] = roc_auc_score(y_test, preds[:,1].detach().cpu().numpy())
            classifier_results['attribution_rnn_acc'] = (prediction_classes.numpy()==y_test).mean()

            # logits
            first_logits = np.stack([sp.special.softmax(i[j]) for i, j in zip(results['logits'], results['start_pos'])])
            first_logits_roc, first_logits_acc = eval_mlp_logit(first_logits)
            classifier_results['first_logits_roc'] = first_logits_roc
            classifier_results['first_logits_acc'] = first_logits_acc

            # fully connected
            for layer in range(results['first_fully_connected'][0].shape[0]):
                layer_roc, layer_acc = eval_mlp_fc(np.stack([i[layer] for i in results['first_fully_connected']]), layer)
                classifier_results[f'first_fully_connected_roc_{layer}'] = layer_roc
                classifier_results[f'first_fully_connected_acc_{layer}'] = layer_acc

            # attention
            for layer in range(results['first_attention'][0].shape[0]):
                layer_roc, layer_acc = eval_mlp_attn(np.stack([i[layer] for i in results['first_attention']]), layer)
                classifier_results[f'first_attention_roc_{layer}'] = layer_roc
                classifier_results[f'first_attention_acc_{layer}'] = layer_acc
            
            all_results[results_file] = classifier_results.copy()
        except Exception as err:
            print(err)

[WindowsPath('results/trex/gpt2_birth_qa_start-0_end-160.pickle'), WindowsPath('results/trex/gpt2_capitals_qa_start-0_end-160.pickle'), WindowsPath('results/trex/gpt2_founders_qa_start-0_end-160.pickle')]


In [32]:
for k,v in all_results.items():
    print(f"Results for {k}:")
    for kk, vv in v.items():
        print(kk, vv)

Results for results\800\gpt2_trivia_qa_start-0_end-800.pickle:
first_logits_roc 0.6475583864118896
first_logits_acc 0.98125
first_fully_connected_roc_0 0.49044585987261147
first_fully_connected_acc_0 0.98125
first_fully_connected_roc_1 0.42462845010615713
first_fully_connected_acc_1 0.98125
first_fully_connected_roc_2 0.4840764331210191
first_fully_connected_acc_2 0.98125
first_fully_connected_roc_3 0.721868365180467
first_fully_connected_acc_3 0.975
first_fully_connected_roc_4 0.6220806794055201
first_fully_connected_acc_4 0.98125
first_fully_connected_roc_5 0.5414012738853503
first_fully_connected_acc_5 0.96875
first_fully_connected_roc_6 0.5477707006369427
first_fully_connected_acc_6 0.98125
first_fully_connected_roc_7 0.6220806794055201
first_fully_connected_acc_7 0.975
first_fully_connected_roc_8 0.732484076433121
first_fully_connected_acc_8 0.975
first_fully_connected_roc_9 0.5902335456475584
first_fully_connected_acc_9 0.98125
first_fully_connected_roc_10 0.5881104033970276
firs

ROC on TRex (3 domains):

Birth: 
- IG: 0.69
- Softmax: 0.55
- FC: 0.68 (layer 11)
- Attention: 0.84 (layer 4) | 0.69 (layer 11)

Capitals:
- IG: 0.31
- Softmax: 0.55
- FC: 0.68 (layer 11)
- Attention: 0.58 (layer 8) | 0.42 (layer 11)

Founders:
- IG: 0.35
- Softmax: 0.47
- FC: 0.44 (layer 11)
- Attention: 0.52 (layer 8) | 0.28 (layer 11)

### Implement 2-layer classifier

In [3]:
class FFHallucinationClassifier_2_layer(torch.nn.Module):
    def __init__(self, input_shape, dropout = dropout_mlp):
        super().__init__()
        self.dropout = dropout
        
        self.linear_relu_stack =torch.nn.Sequential(
            # First hidden layer
            torch.nn.Linear(input_shape, 256),
            torch.nn.ReLU(),
            torch.nn.Dropout(self.dropout),

            # Second hidden layer
            torch.nn.Linear(256, 256),
            torch.nn.ReLU(),
            torch.nn.Dropout(self.dropout),

            torch.nn.Linear(256, 2)
            )

    def forward(self, x):
        logits = self.linear_relu_stack(x)
        return logits
    
def gen_classifier_roc_2_layer(inputs):
    X_train, X_test, y_train, y_test = train_test_split(inputs, correct.astype(int), test_size=0.2, random_state=123)
    classifier_model = FFHallucinationClassifier_2_layer(X_train.shape[1]).to(device)
    X_train = torch.tensor(X_train).to(device)
    y_train = torch.tensor(y_train).to(torch.long).to(device)
    X_test = torch.tensor(X_test).to(device)
    y_test = torch.tensor(y_test).to(torch.long).to(device)

    optimizer = torch.optim.AdamW(classifier_model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    for _ in tqdm(range(1001)):
        optimizer.zero_grad()
        sample = torch.randperm(X_train.shape[0])[:batch_size]
        pred = classifier_model(X_train[sample])
        loss = torch.nn.functional.cross_entropy(pred, y_train[sample])
        loss.backward()
        optimizer.step()
    classifier_model.eval()
    with torch.no_grad():
        pred = torch.nn.functional.softmax(classifier_model(X_test), dim=1)
        prediction_classes = (pred[:,1]>0.5).type(torch.long).cpu()
        return roc_auc_score(y_test.cpu(), pred[:,1].cpu()), (prediction_classes.numpy()==y_test.cpu().numpy()).mean()

In [4]:
inference_results = list(Path("./results/1000_samples").rglob("*.pickle"))
print(inference_results)
all_results = {}

[PosixPath('results/1000_samples/open_llama_7b_trivia_qa_start-0_end-1000_4_28.pickle')]


In [5]:
for results_file in inference_results:
    if results_file not in all_results.keys():
        try:
            del results
        except:
            pass
        try:
            classifier_results = {}
            with open(results_file, "rb") as infile:
                results = pickle.loads(infile.read())
            correct = np.array(results['correct'])

            # logits
            first_logits = np.stack([sp.special.softmax(i[j]) for i, j in zip(results['logits'], results['start_pos'])])
            first_logits_roc, first_logits_acc = gen_classifier_roc_2_layer(first_logits)
            classifier_results['first_logits_roc'] = first_logits_roc
            classifier_results['first_logits_acc'] = first_logits_acc

            # fully connected
            layer = 31
            layer_roc, layer_acc = gen_classifier_roc_2_layer(np.stack([i[layer] for i in results['first_fully_connected']]))
            classifier_results[f'first_fully_connected_roc_{layer}'] = layer_roc
            classifier_results[f'first_fully_connected_acc_{layer}'] = layer_acc

            # attention
            layer_roc, layer_acc = gen_classifier_roc_2_layer(np.stack([i[layer] for i in results['first_attention']]))
            classifier_results[f'first_attention_roc_{layer}'] = layer_roc
            classifier_results[f'first_attention_acc_{layer}'] = layer_acc

            first_fc_features = np.stack([i[layer] for i in results['first_fully_connected']])
            first_attn_features = np.stack([i[layer] for i in results['first_attention']])
            combined_features = np.concatenate([first_fc_features, first_attn_features], axis=1)

            layer_roc, layer_acc = gen_classifier_roc_2_layer(combined_features)
            classifier_results[f'first_fc+attn_roc_31'] = layer_roc
            classifier_results[f'first_fc+attn_acc_31'] = layer_acc
            
            all_results[results_file] = classifier_results.copy()
        except Exception as err:
            print(err)

100%|██████████| 1001/1001 [00:03<00:00, 290.22it/s]


In [6]:
for k,v in all_results.items():
    print(f"Results for {k}:")
    for kk, vv in v.items():
        print(kk, vv)

Results for results/1000_samples/open_llama_7b_trivia_qa_start-0_end-1000_4_28.pickle:
first_logits_roc 0.6020925466508467
first_logits_acc 0.625
first_fully_connected_roc_31 0.6692374069679646
first_fully_connected_acc_31 0.665
first_attention_roc_31 0.6404918563261784
first_attention_acc_31 0.62
first_fc+attn_roc_31 0.6532197173983388
first_fc+attn_acc_31 0.64


AUROC of 2-layer NN on GPT-2 and TriviaQA (800 samples):
- Softmax probability: 0.35
- Fully-connected activation: 0.84 (layer 11)
- Self-attention score: 0.75 (layer 6) | 0.67 (layer 11)

### Implement 4-layer classifier

In [41]:
class FFHallucinationClassifier_4_layer(torch.nn.Module):
    def __init__(self, input_shape, dropout = dropout_mlp):
        super().__init__()
        self.dropout = dropout
        
        self.linear_relu_stack =torch.nn.Sequential(
            # First hidden layer
            torch.nn.Linear(input_shape, 256),
            torch.nn.ReLU(),
            torch.nn.Dropout(self.dropout),

            # Second hidden layer
            torch.nn.Linear(256, 256),
            torch.nn.ReLU(),
            torch.nn.Dropout(self.dropout),

            # Third
            torch.nn.Linear(256, 128),
            torch.nn.ReLU(),
            torch.nn.Dropout(self.dropout),

            # Fourth
            torch.nn.Linear(128, 64),
            torch.nn.ReLU(),
            torch.nn.Dropout(self.dropout),

            torch.nn.Linear(64, 2)
            )

    def forward(self, x):
        logits = self.linear_relu_stack(x)
        return logits
    
def gen_classifier_roc_4_layer(inputs):
    X_train, X_test, y_train, y_test = train_test_split(inputs, correct.astype(int), test_size=0.2, random_state=123)
    classifier_model = FFHallucinationClassifier_4_layer(X_train.shape[1]).to(device)
    X_train = torch.tensor(X_train).to(device)
    y_train = torch.tensor(y_train).to(torch.long).to(device)
    X_test = torch.tensor(X_test).to(device)
    y_test = torch.tensor(y_test).to(torch.long).to(device)

    optimizer = torch.optim.AdamW(classifier_model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    for _ in tqdm(range(1001)):
        optimizer.zero_grad()
        sample = torch.randperm(X_train.shape[0])[:batch_size]
        pred = classifier_model(X_train[sample])
        loss = torch.nn.functional.cross_entropy(pred, y_train[sample])
        loss.backward()
        optimizer.step()
    classifier_model.eval()
    with torch.no_grad():
        pred = torch.nn.functional.softmax(classifier_model(X_test), dim=1)
        prediction_classes = (pred[:,1]>0.5).type(torch.long).cpu()
        return roc_auc_score(y_test.cpu(), pred[:,1].cpu()), (prediction_classes.numpy()==y_test.cpu().numpy()).mean()

In [45]:
inference_results = list(Path("./results/1000_samples").rglob("*.pickle"))
print(inference_results)
all_results = {}

[PosixPath('results/1000_samples/open_llama_7b_trivia_qa_start-0_end-1000_4_28.pickle')]


In [46]:
for results_file in inference_results:
    if results_file not in all_results.keys():
        try:
            del results
        except:
            pass
        try:
            classifier_results = {}
            with open(results_file, "rb") as infile:
                results = pickle.loads(infile.read())
            correct = np.array(results['correct'])

            # logits
            first_logits = np.stack([sp.special.softmax(i[j]) for i, j in zip(results['logits'], results['start_pos'])])
            first_logits_roc, first_logits_acc = gen_classifier_roc_4_layer(first_logits)
            classifier_results['first_logits_roc'] = first_logits_roc
            classifier_results['first_logits_acc'] = first_logits_acc

            # fully connected
            layer = 31
            layer_roc, layer_acc = gen_classifier_roc_4_layer(np.stack([i[layer] for i in results['first_fully_connected']]))
            classifier_results[f'first_fully_connected_roc_{layer}'] = layer_roc
            classifier_results[f'first_fully_connected_acc_{layer}'] = layer_acc

            # attention
            layer_roc, layer_acc = gen_classifier_roc_4_layer(np.stack([i[layer] for i in results['first_attention']]))
            classifier_results[f'first_attention_roc_{layer}'] = layer_roc
            classifier_results[f'first_attention_acc_{layer}'] = layer_acc

            first_fc_features = np.stack([i[layer] for i in results['first_fully_connected']])
            first_attn_features = np.stack([i[layer] for i in results['first_attention']])
            combined_features = np.concatenate([first_fc_features, first_attn_features], axis=1)

            layer_roc, layer_acc = gen_classifier_roc_4_layer(combined_features)
            classifier_results[f'fc+attention_roc_{layer}'] = layer_roc
            classifier_results[f'fc+attention_acc_{layer}'] = layer_acc
            
            all_results[results_file] = classifier_results.copy()
        except Exception as err:
            print(err)

100%|██████████| 1001/1001 [00:04<00:00, 228.51it/s]


In [47]:
for k,v in all_results.items():
    print(f"Results for {k}:")
    for kk, vv in v.items():
        print(kk, vv)

Results for results/1000_samples/open_llama_7b_trivia_qa_start-0_end-1000_4_28.pickle:
first_logits_roc 0.6229101499298888
first_logits_acc 0.635
first_fully_connected_roc_31 0.6544062129220148
first_fully_connected_acc_31 0.66
first_attention_roc_31 0.642541257685255
first_attention_acc_31 0.625
fc+attention_roc_31 0.641354762161579
fc+attention_acc_31 0.635


AUROC of 5-layer NN on GPT-2 and TriviaQA (800 samples):
- Softmax probability: 0.36
- Fully-connected activation: 0.75 (layer 11)
- Self-attention score: 0.74 (layer 6) | 0.7 (layer 11)

### Correctly predicted on True & False samples

In [ ]:
def predict_by_class(inputs, y_true, checkpoint_path):
    X_train, X_test, y_train, y_test = train_test_split(inputs, y_true.astype(int), test_size=0.2, random_state=123)
    
    classifier_model = FFHallucinationClassifier(X_train.shape[1]).to(device)
    classifier_model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    
    X_test = torch.tensor(X_test).to(device)
    y_test = torch.tensor(y_test).to(torch.long).to(device)
    
    classifier_model.eval()
    with torch.no_grad():
        pred = torch.nn.functional.softmax(classifier_model(X_test), dim=1)
        prediction_classes = (pred[:,1]>0.5).type(torch.long).cpu().numpy()
        y_test_np = y_test.cpu().numpy()
    
    # Class 0 (False samples)
    class_0_total = np.sum(y_test_np == 0)
    class_0_correct = np.sum((y_test_np == 0) & (prediction_classes == 0))
    
    # Class 1 (True samples)
    class_1_total = np.sum(y_test_np == 1)
    class_1_correct = np.sum((y_test_np == 1) & (prediction_classes == 1))
    
    return {
        'class_0_correct': class_0_correct,
        'class_0_total': class_0_total,
        'class_1_correct': class_1_correct,
        'class_1_total': class_1_total,
        'roc': roc_auc_score(y_test_np, pred[:,1].cpu()),
        'accuracy': (prediction_classes == y_test_np).mean()
    }

inference_results = list(Path("./results/800").rglob("*.pickle"))
with open(inference_results[0], "rb") as infile:
    results = pickle.loads(infile.read())
correct = np.array(results['correct'])

In [13]:
first_logits = np.stack([sp.special.softmax(i[j]) for i, j in zip(results['logits'], results['start_pos'])])
analysis = predict_by_class(first_logits, correct, "./models/mlp_logit.pt")

print("Prediction for Logits:")
print(f"Class 0 (False): {analysis['class_0_correct']}/{analysis['class_0_total']} correct ({100*analysis['class_0_correct']/analysis['class_0_total']:.1f}%)")
print(f"Class 1 (True): {analysis['class_1_correct']}/{analysis['class_1_total']} correct ({100*analysis['class_1_correct']/analysis['class_1_total']:.1f}%)")
print(f"Overall Accuracy: {analysis['accuracy']:.4f}, ROC: {analysis['roc']:.4f}")

Prediction for Logits:
Class 0 (False): 157/157 correct (100.0%)
Class 1 (True): 0/3 correct (0.0%)
Overall Accuracy: 0.9812, ROC: 0.6476


In [14]:
fc_layer_11 = np.stack([i[11] for i in results['first_fully_connected']])
analysis_fc = predict_by_class(fc_layer_11, correct, "./models/mlp_fc_11.pt")

print("Prediction for FC last layer:")
print(f"Class 0 (False): {analysis_fc['class_0_correct']}/{analysis_fc['class_0_total']} correct ({100*analysis_fc['class_0_correct']/analysis_fc['class_0_total']:.1f}%)")
print(f"Class 1 (True): {analysis_fc['class_1_correct']}/{analysis_fc['class_1_total']} correct ({100*analysis_fc['class_1_correct']/analysis_fc['class_1_total']:.1f}%)")
print(f"Overall Accuracy: {analysis_fc['accuracy']:.4f}, ROC: {analysis_fc['roc']:.4f}")

Prediction for FC last layer:
Class 0 (False): 157/157 correct (100.0%)
Class 1 (True): 0/3 correct (0.0%)
Overall Accuracy: 0.9812, ROC: 0.8068


In [15]:
attn_layer_11 = np.stack([i[11] for i in results['first_attention']])
analysis_attn = predict_by_class(attn_layer_11, correct, "./models/mlp_attn_11.pt")

print("Prediction for Attention last layer:")
print(f"Class 0 (False): {analysis_attn['class_0_correct']}/{analysis_attn['class_0_total']} correct ({100*analysis_attn['class_0_correct']/analysis_attn['class_0_total']:.1f}%)")
print(f"Class 1 (True): {analysis_attn['class_1_correct']}/{analysis_attn['class_1_total']} correct ({100*analysis_attn['class_1_correct']/analysis_attn['class_1_total']:.1f}%)")
print(f"Overall Accuracy: {analysis_attn['accuracy']:.4f}, ROC: {analysis_attn['roc']:.4f}")

Prediction for Attention last layer:
Class 0 (False): 157/157 correct (100.0%)
Class 1 (True): 0/3 correct (0.0%)
Overall Accuracy: 0.9812, ROC: 0.6539


True/False prediction of Open-Llama-7B

In [ ]:
inference_results = list(Path("./results/1000_samples").rglob("*.pickle"))
with open(inference_results[0], "rb") as infile:
    results = pickle.loads(infile.read())
correct = np.array(results['correct'])

In [11]:
fc_layer_31 = np.stack([i[31] for i in results['first_fully_connected']])
analysis_fc = predict_by_class(fc_layer_31, correct, "./models/mlp_fc_31.pt")

print("Prediction for FC last layer:")
print(f"Class 0 (False): {analysis_fc['class_0_correct']}/{analysis_fc['class_0_total']} correct ({100*analysis_fc['class_0_correct']/analysis_fc['class_0_total']:.1f}%)")
print(f"Class 1 (True): {analysis_fc['class_1_correct']}/{analysis_fc['class_1_total']} correct ({100*analysis_fc['class_1_correct']/analysis_fc['class_1_total']:.1f}%)")
print(f"Overall Accuracy: {analysis_fc['accuracy']:.4f}, ROC: {analysis_fc['roc']:.4f}")

Prediction for FC last layer:
Class 0 (False): 85/127 correct (66.9%)
Class 1 (True): 43/73 correct (58.9%)
Overall Accuracy: 0.6400, ROC: 0.6665


### n-layer Transformer

In [3]:
import torch
import torch.nn as nn

class TransformerHallucinationClassifier(nn.Module):
    def __init__(self, d_model, n_head=8, num_layers=3, dim_feedforward=256, dropout=dropout_mlp):
        super().__init__()
        self.d_model = d_model
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=n_head, 
            dim_feedforward=dim_feedforward, 
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )
        self.transformer_extractor = nn.TransformerEncoder(
            encoder_layer, 
            num_layers=num_layers
        )
        
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(d_model, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 2)
        )

    def forward(self, x, padding_mask=None):
        if x.dim() == 2:
            x = x.unsqueeze(1)

        transformer_out = self.transformer_extractor(x, src_key_padding_mask=padding_mask)

        if padding_mask is not None:
            expanded_mask = padding_mask.unsqueeze(-1).expand_as(transformer_out)
            transformer_out = transformer_out.masked_fill(expanded_mask, 0.0)
            sum_embeddings = torch.sum(transformer_out, dim=1)
            valid_tokens = (~padding_mask).sum(dim=1, keepdim=True).clamp(min=1)
            pooled_out = sum_embeddings / valid_tokens
        else:
            pooled_out = transformer_out.mean(dim=1)

        logits = self.linear_relu_stack(pooled_out)
        return logits

In [4]:
batch_size = 4
def gen_classifier_roc(inputs, save_name="mlp_model.pt"):
    X_train, X_test, y_train, y_test = train_test_split(inputs, correct.astype(int), test_size=0.2, random_state=123)
    classifier_model = TransformerHallucinationClassifier(X_train.shape[1]).to(device)
    X_train = torch.tensor(X_train).to(device)
    y_train = torch.tensor(y_train).to(torch.long).to(device)
    X_test = torch.tensor(X_test).to(device)
    y_test = torch.tensor(y_test).to(torch.long).to(device)

    optimizer = torch.optim.AdamW(classifier_model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    for _ in tqdm(range(1001)):
        optimizer.zero_grad()
        sample = torch.randperm(X_train.shape[0])[:batch_size]
        pred = classifier_model(X_train[sample])
        loss = torch.nn.functional.cross_entropy(pred, y_train[sample])
        loss.backward()
        optimizer.step()
    torch.save(classifier_model.state_dict(), f"./models/{save_name}")
    classifier_model.eval()
    with torch.no_grad():
        pred = torch.nn.functional.softmax(classifier_model(X_test), dim=1)
        prediction_classes = (pred[:,1]>0.5).type(torch.long).cpu()
        return roc_auc_score(y_test.cpu(), pred[:,1].cpu()), (prediction_classes.numpy()==y_test.cpu().numpy()).mean()

In [5]:
inference_results = list(Path("./results/1000_samples").rglob("*.pickle"))
print(inference_results)
os.makedirs("./models/", exist_ok=True)

[PosixPath('results/1000_samples/open_llama_7b_trivia_qa_start-0_end-1000_4_28.pickle')]


In [6]:
all_results = {}

In [ ]:
for results_file in inference_results:
    if results_file not in all_results.keys():
        try:
            del results
        except:
            pass
        try:
            classifier_results = {}
            with open(results_file, "rb") as infile:
                results = pickle.loads(infile.read())
            correct = np.array(results['correct'])

            # fully connected
            layer_roc, layer_acc = gen_classifier_roc(np.stack([i[31] for i in results['first_fully_connected']]), save_name=f"transformer2_fc_31.pt")
            classifier_results[f'first_fully_connected_roc_31'] = layer_roc
            classifier_results[f'first_fully_connected_acc_31'] = layer_acc

            # attention
            layer_roc, layer_acc = gen_classifier_roc(np.stack([i[31] for i in results['first_attention']]), save_name=f"transformer2_attn_31.pt")
            classifier_results[f'first_attention_roc_31'] = layer_roc
            classifier_results[f'first_attention_acc_31'] = layer_acc
            
            all_results[results_file] = classifier_results.copy()
        except Exception as err:
            print(err)

/data/share/project/RSML/llm-hallucinations-factual-qa/.venv/lib/python3.10/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
100%|██████████| 1001/1001 [01:40<00:00,  9.93it/s]
/data/share/project/RSML/llm-hallucinations-factual-qa/.venv/lib/python3.10/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
100%|██████████| 1001/1001 [00:18<00:00, 54.39it/s]


In [7]:
for results_file in inference_results:
    if results_file not in all_results.keys():
        try:
            del results
        except:
            pass
        try:
            classifier_results = {}
            with open(results_file, "rb") as infile:
                results = pickle.loads(infile.read())
            correct = np.array(results['correct'])

            # fully connected
            layer_roc, layer_acc = gen_classifier_roc(np.stack([i[31] for i in results['first_fully_connected']]), save_name=f"transformer3_fc_31.pt")
            classifier_results[f'first_fully_connected_roc_31'] = layer_roc
            classifier_results[f'first_fully_connected_acc_31'] = layer_acc

            # attention
            layer_roc, layer_acc = gen_classifier_roc(np.stack([i[31] for i in results['first_attention']]), save_name=f"transformer3_attn_31.pt")
            classifier_results[f'first_attention_roc_31'] = layer_roc
            classifier_results[f'first_attention_acc_31'] = layer_acc
            
            all_results[results_file] = classifier_results.copy()
        except Exception as err:
            print(err)

/data/share/project/RSML/llm-hallucinations-factual-qa/.venv/lib/python3.10/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
100%|██████████| 1001/1001 [02:26<00:00,  6.84it/s]
/data/share/project/RSML/llm-hallucinations-factual-qa/.venv/lib/python3.10/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
100%|██████████| 1001/1001 [00:26<00:00, 37.20it/s]


2-layer Transformer

In [7]:
for k,v in all_results.items():
    for kk, vv in v.items():
        print(kk, vv)

first_fully_connected_roc_31 0.572106568870672
first_fully_connected_acc_31 0.635
first_attention_roc_31 0.6304605759896451
first_attention_acc_31 0.63


3-layer Transformer

In [9]:
for k,v in all_results.items():
    for kk, vv in v.items():
        print(kk, vv)

first_fully_connected_roc_31 0.49482256498759575
first_fully_connected_acc_31 0.635
first_attention_roc_31 0.5736166540826232
first_attention_acc_31 0.65


### Concatenate FC activation and attention score together

In [2]:
with open("./results/100_samples/open_llama_7b_trivia_qa_start-0_end-100_4_28.pickle", "rb") as infile:
    results = pickle.loads(infile.read())
correct = np.array(results['correct'])
batch_size = 4

first_fc_features = np.stack([i[31] for i in results['first_fully_connected']])
first_attn_features = np.stack([i[31] for i in results['first_attention']])
print(f"first_fc_features shape: {first_fc_features.shape}")
print(f"first_attn_features shape: {first_attn_features.shape}")

combined_features = np.concatenate([first_fc_features, first_attn_features], axis=1)
print(f"combined_features shape: {combined_features.shape}")

X_train, X_test, y_train, y_test = train_test_split(combined_features, correct.astype(int), test_size=0.2, random_state=1234)
X_train = torch.tensor(X_train)
# sample = torch.randperm(X_train.shape[0])[:batch_size]
print("Sample shape:", X_train.shape)

first_fc_features shape: (100, 11008)
first_attn_features shape: (100, 4096)
combined_features shape: (100, 15104)
Sample shape: torch.Size([80, 15104])


In [10]:
def gen_classifier_roc(inputs):
    X_train, X_test, y_train, y_test = train_test_split(inputs, correct.astype(int), test_size=0.2, random_state=123)
    classifier_model = FFHallucinationClassifier(X_train.shape[1]).to(device)
    X_train = torch.tensor(X_train).to(device)
    y_train = torch.tensor(y_train).to(torch.long).to(device)
    X_test = torch.tensor(X_test).to(device)
    y_test = torch.tensor(y_test).to(torch.long).to(device)

    optimizer = torch.optim.AdamW(classifier_model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    for _ in tqdm(range(1001)):
        optimizer.zero_grad()
        sample = torch.randperm(X_train.shape[0])[:batch_size]
        pred = classifier_model(X_train[sample])
        loss = torch.nn.functional.cross_entropy(pred, y_train[sample])
        loss.backward()
        optimizer.step()
    classifier_model.eval()
    with torch.no_grad():
        pred = torch.nn.functional.softmax(classifier_model(X_test), dim=1)
        prediction_classes = (pred[:,1]>0.5).type(torch.long).cpu()
        return roc_auc_score(y_test.cpu(), pred[:,1].cpu()), (prediction_classes.numpy()==y_test.cpu().numpy()).mean()

In [15]:
all_results = {}
batch_size = 128

In [16]:
for results_file in inference_results:
    if results_file not in all_results.keys():
        try:
            del results
        except:
            pass
        try:
            classifier_results = {}
            with open(results_file, "rb") as infile:
                results = pickle.loads(infile.read())
            correct = np.array(results['correct'])
            
            # combine features at last layer
            first_fc_features = np.stack([i[31] for i in results['first_fully_connected']])
            first_attn_features = np.stack([i[31] for i in results['first_attention']])
            combined_features = np.concatenate([first_fc_features, first_attn_features], axis=1)

            layer_roc, layer_acc = gen_classifier_roc(combined_features)
            classifier_results[f'first_fc+attn_roc_31'] = layer_roc
            classifier_results[f'first_fc+attn_acc_31'] = layer_acc
            
            all_results[results_file] = classifier_results.copy()
        except Exception as err:
            print(err)

100%|██████████| 1001/1001 [00:02<00:00, 417.32it/s]


In [17]:
for k,v in all_results.items():
    for kk, vv in v.items():
        print(kk, vv)

first_fc+attn_roc_31 0.6528961277100637
first_fc+attn_acc_31 0.635


### GNN Classifier

In [4]:
import pickle
import numpy as np
from sklearn.neighbors import kneighbors_graph
from sknetwork.gnn import GNNClassifier

results_file = "./results/1000_samples/open_llama_7b_trivia_qa_start-0_end-1000_4_28.pickle"
with open(results_file, "rb") as infile:
    results = pickle.loads(infile.read())
correct = np.array(results['correct']).astype(int)
first_fc_features = np.stack([i[31] for i in results['first_fully_connected']])

labels = np.full(1000, -1)
labels[:800] = correct[:800]

adjacency = kneighbors_graph(
                first_fc_features, 
                n_neighbors=5, 
                mode='connectivity',
                include_self=False
            )
gnn = GNNClassifier(dims=[1024, 512, 256, 128, 64], learning_rate=learning_rate)

print("Training the GNN...")
gnn.fit(adjacency, first_fc_features, labels)


Training the GNN...


GNNClassifier(
    Convolution(layer_type: conv, out_channels: 1024, activation: ReLu, use_bias: True, normalization: both, self_embeddings: True)
    Convolution(layer_type: conv, out_channels: 512, activation: ReLu, use_bias: True, normalization: both, self_embeddings: True)
    Convolution(layer_type: conv, out_channels: 256, activation: ReLu, use_bias: True, normalization: both, self_embeddings: True)
    Convolution(layer_type: conv, out_channels: 128, activation: ReLu, use_bias: True, normalization: both, self_embeddings: True)
    Convolution(layer_type: conv, out_channels: 64, activation: Cross entropy, use_bias: True, normalization: both, self_embeddings: True)
)

In [5]:
predictions = gnn.labels_
print(len(predictions))

1000


In [6]:
from sklearn.metrics import roc_auc_score
print(f"Accuracy: {(predictions[800:] == correct[800:]).mean():.4f}")
print(f"ROC-AUC: {roc_auc_score(correct[800:], predictions[800:]):.4f}")

Accuracy: 0.6700
ROC-AUC: 0.5257


### Deep Neural Networks with a Residual block

3-layer DNN

In [3]:
import torch
import torch.nn as nn
torch.autograd.set_detect_anomaly(True)

class ResidualBlock(nn.Module):
    def __init__(self, hidden_dim):
        super(ResidualBlock, self).__init__()
        self.fc1 = nn.Linear(hidden_dim, hidden_dim)
        self.relu = nn.ReLU()
        # self.fc2 = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x):
        identity = x
        
        out = self.fc1(x)
        # out = self.relu(out)
        # out = self.fc2(out)
        
        out = out + identity
        out = self.relu(out)
        
        return out

class DNNClassifier(nn.Module):
    def __init__(self, input_size, hidden_size=256):
        super(DNNClassifier, self).__init__()
        
        self.layer1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()

        self.res_block = ResidualBlock(hidden_size)

        self.layer3 = nn.Linear(hidden_size, 2)
        
    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        
        x = self.res_block(x)

        x = self.layer3(x)
        
        return x

In [4]:
def gen_classifier_roc(inputs):
    X_train, X_test, y_train, y_test = train_test_split(inputs, correct.astype(int), test_size=0.2, random_state=123)
    classifier_model = DNNClassifier(X_train.shape[1]).to(device)
    X_train = torch.tensor(X_train).to(device)
    y_train = torch.tensor(y_train).to(torch.long).to(device)
    X_test = torch.tensor(X_test).to(device)
    y_test = torch.tensor(y_test).to(torch.long).to(device)

    optimizer = torch.optim.AdamW(classifier_model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    for _ in tqdm(range(1001)):
        optimizer.zero_grad()
        sample = torch.randperm(X_train.shape[0])[:batch_size]
        pred = classifier_model(X_train[sample])
        loss = torch.nn.functional.cross_entropy(pred, y_train[sample])
        loss.backward()
        optimizer.step()
    classifier_model.eval()
    with torch.no_grad():
        pred = torch.nn.functional.softmax(classifier_model(X_test), dim=1)
        prediction_classes = (pred[:,1]>0.5).type(torch.long).cpu()
        return roc_auc_score(y_test.cpu(), pred[:,1].cpu()), (prediction_classes.numpy()==y_test.cpu().numpy()).mean()

In [5]:
all_results = {}

In [6]:
for results_file in inference_results:
    if results_file not in all_results.keys():
        try:
            del results
        except:
            pass
        try:
            classifier_results = {}
            with open(results_file, "rb") as infile:
                results = pickle.loads(infile.read())
            correct = np.array(results['correct'])

            # logits
            first_logits = np.stack([sp.special.softmax(i[j]) for i, j in zip(results['logits'], results['start_pos'])])
            first_logits_roc, first_logits_acc = gen_classifier_roc(first_logits)
            classifier_results['first_logits_roc'] = first_logits_roc
            classifier_results['first_logits_acc'] = first_logits_acc

            # fully connected
            for layer in range(results['first_fully_connected'][0].shape[0]):
                layer_roc, layer_acc = gen_classifier_roc(np.stack([i[layer] for i in results['first_fully_connected']]))
                classifier_results[f'first_fully_connected_roc_{layer}'] = layer_roc
                classifier_results[f'first_fully_connected_acc_{layer}'] = layer_acc

            # attention
            for layer in range(results['first_attention'][0].shape[0]):
                layer_roc, layer_acc = gen_classifier_roc(np.stack([i[layer] for i in results['first_attention']]))
                classifier_results[f'first_attention_roc_{layer}'] = layer_roc
                classifier_results[f'first_attention_acc_{layer}'] = layer_acc

            pooled_embeddings = []
            for i in range(len(results['generated_embeddings'])):
                embeddings = results['generated_embeddings'][i]
                # print(f"Embeddings shape for sample {i}: {embeddings}")
                emb_sequence = torch.tensor(embeddings, dtype=torch.float32)
                pooled_emb = torch.mean(emb_sequence, dim=0)
                pooled_embeddings.append(pooled_emb)

            pooled_embeddings_tensor = torch.stack(pooled_embeddings)
            context_emb_roc, context_emb_acc = gen_classifier_roc(pooled_embeddings_tensor)
            classifier_results['context_emb_roc'] = context_emb_roc
            classifier_results['context_emb_acc'] = context_emb_acc
            
            all_results[results_file] = classifier_results.copy()
        except Exception as err:
            print(err)

100%|██████████| 1001/1001 [00:08<00:00, 123.96it/s]
/tmp/ipykernel_1047797/1166460029.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_train = torch.tensor(X_train).to(device)
/tmp/ipykernel_1047797/1166460029.py:6: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_test = torch.tensor(X_test).to(device)
100%|██████████| 1001/1001 [00:08<00:00, 125.07it/s]


In [7]:
for k,v in all_results.items():
    for kk, vv in v.items():
        print(kk, vv)

first_logits_roc 0.700392871912478
first_logits_acc 0.668
first_fully_connected_roc_0 0.6688383230781441
first_fully_connected_acc_0 0.6565
first_fully_connected_roc_1 0.6760404526485959
first_fully_connected_acc_1 0.6545
first_fully_connected_roc_2 0.6932434803313408
first_fully_connected_acc_2 0.6735
first_fully_connected_roc_3 0.7074075466496235
first_fully_connected_acc_3 0.6675
first_fully_connected_roc_4 0.7008163423019245
first_fully_connected_acc_4 0.63
first_fully_connected_roc_5 0.70736786261806
first_fully_connected_acc_5 0.6715
first_fully_connected_roc_6 0.7201210571826009
first_fully_connected_acc_6 0.687
first_fully_connected_roc_7 0.7201952036626272
first_fully_connected_acc_7 0.685
first_fully_connected_roc_8 0.7135241090934914
first_fully_connected_acc_8 0.683
first_fully_connected_roc_9 0.7057136650918372
first_fully_connected_acc_9 0.6475
first_fully_connected_roc_10 0.7153015359808849
first_fully_connected_acc_10 0.6775
first_fully_connected_roc_11 0.71968348851878

5-layer DNN

In [2]:
import torch
import torch.nn as nn

class ResidualBlock(nn.Module):
    def __init__(self, hidden_dim):
        super(ResidualBlock, self).__init__()
        self.fc1 = nn.Linear(hidden_dim, hidden_dim)
        self.relu = nn.ReLU()
        # self.fc2 = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x):
        identity = x
        
        out = self.fc1(x)
        # out = self.relu(out)

        # out = self.fc2(out)

        out += identity
        out = self.relu(out)
        
        return out

class DNNClassifier(nn.Module):
    def __init__(self, input_size, hidden_size=256):
        super(DNNClassifier, self).__init__()
        
        self.layer1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()

        self.res_block = nn.Sequential(
            ResidualBlock(hidden_size),
            ResidualBlock(hidden_size),
            ResidualBlock(hidden_size),
        )

        self.layer5 = nn.Linear(hidden_size, 2)
        
    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        
        x = self.res_block(x)

        x = self.layer5(x)
        
        return x

8-layer DNN

In [2]:
import torch
import torch.nn as nn

class ResidualBlock(nn.Module):
    def __init__(self, hidden_dim):
        super(ResidualBlock, self).__init__()
        self.fc1 = nn.Linear(hidden_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x):
        identity = x
        
        out = self.fc1(x)
        out = self.relu(out)

        out = self.fc2(out)

        out += identity
        out = self.relu(out)
        
        return out

class DNNClassifier(nn.Module):
    def __init__(self, input_size, hidden_size=256):
        super(DNNClassifier, self).__init__()
        
        self.layer1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()

        self.res_block = nn.Sequential(
            ResidualBlock(hidden_size),
            ResidualBlock(hidden_size),
            ResidualBlock(hidden_size),
        )

        self.layer5 = nn.Linear(hidden_size, 2)
        
    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        
        x = self.res_block(x)

        x = self.layer5(x)
        
        return x

In [3]:
def gen_classifier_roc(inputs):
    X_train, X_test, y_train, y_test = train_test_split(inputs, correct.astype(int), test_size=0.2, random_state=123)
    classifier_model = DNNClassifier(X_train.shape[1]).to(device)
    X_train = torch.tensor(X_train).to(device)
    y_train = torch.tensor(y_train).to(torch.long).to(device)
    X_test = torch.tensor(X_test).to(device)
    y_test = torch.tensor(y_test).to(torch.long).to(device)

    optimizer = torch.optim.AdamW(classifier_model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    for _ in tqdm(range(1001)):
        optimizer.zero_grad()
        sample = torch.randperm(X_train.shape[0])[:batch_size]
        pred = classifier_model(X_train[sample])
        loss = torch.nn.functional.cross_entropy(pred, y_train[sample])
        loss.backward()
        optimizer.step()
    classifier_model.eval()
    with torch.no_grad():
        pred = torch.nn.functional.softmax(classifier_model(X_test), dim=1)
        prediction_classes = (pred[:,1]>0.5).type(torch.long).cpu()
        return roc_auc_score(y_test.cpu(), pred[:,1].cpu()), (prediction_classes.numpy()==y_test.cpu().numpy()).mean()

In [5]:
all_results = {}

In [ ]:
for results_file in inference_results:
    if results_file not in all_results.keys():
        try:
            del results
        except:
            pass
        try:
            classifier_results = {}
            with open(results_file, "rb") as infile:
                results = pickle.loads(infile.read())
            correct = np.array(results['correct'])

            # logits
            first_logits = np.stack([sp.special.softmax(i[j]) for i, j in zip(results['logits'], results['start_pos'])])
            first_logits_roc, first_logits_acc = gen_classifier_roc(first_logits)
            classifier_results['first_logits_roc'] = first_logits_roc
            classifier_results['first_logits_acc'] = first_logits_acc

            layer = 31
            layer_roc, layer_acc = gen_classifier_roc(np.stack([i[layer] for i in results['first_fully_connected']]))
            classifier_results[f'first_fully_connected_roc_{layer}'] = layer_roc
            classifier_results[f'first_fully_connected_acc_{layer}'] = layer_acc

            # attention
            layer_roc, layer_acc = gen_classifier_roc(np.stack([i[layer] for i in results['first_attention']]))
            classifier_results[f'first_attention_roc_{layer}'] = layer_roc
            classifier_results[f'first_attention_acc_{layer}'] = layer_acc
         
            all_results[results_file] = classifier_results.copy()
        except Exception as err:
            print(err)

100%|██████████| 1001/1001 [00:05<00:00, 176.97it/s]


In [7]:
for k,v in all_results.items():
    for kk, vv in v.items():
        print(kk, vv)

first_logits_roc 0.6015532305037212
first_logits_acc 0.585
first_fully_connected_roc_31 0.6475029662388093
first_fully_connected_acc_31 0.66
first_attention_roc_31 0.6505770682774242
first_attention_acc_31 0.595


### CNN classifier

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ResidualBlock1D(nn.Module):
    def __init__(self, channels):
        super(ResidualBlock1D, self).__init__()
        self.conv1 = nn.Conv1d(channels, channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(channels)
        
        self.conv2 = nn.Conv1d(channels, channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(channels)
        
        # self.conv3 = nn.Conv1d(channels, channels, kernel_size=3, padding=1)
        # self.bn3 = nn.BatchNorm1d(channels)

    def forward(self, x):
        residual = x  
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.bn2(self.conv2(out))
        # out = self.bn3(self.conv3(out))
        
        out += residual  
        out = F.relu(out)
        
        return out

class CNNClassifier(nn.Module):
    def __init__(self, num_classes=2): # Assuming binary or multi-class
        super(CNNClassifier, self).__init__()

        self.layer1 = nn.Sequential(
            # Using a slightly larger kernel and stride to quickly reduce the massive 11008 length
            nn.Conv1d(in_channels=1, out_channels=32, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4, stride=4)
        )

        self.res_block = nn.Sequential(
            ResidualBlock1D(channels=32),
            ResidualBlock1D(channels=32),
            ResidualBlock1D(channels=32)
        )

        self.layer3 = nn.Sequential(
            nn.Conv1d(in_channels=32, out_channels=256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4, stride=4)
        )

        # Global Average Pooling
        # This collapses whatever sequence length is left down to exactly 1.
        # This prevents the final Linear layer from having millions of parameters.
        self.global_pool = nn.AdaptiveAvgPool1d(1)

        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        # 1. Reshape input from (Batch, Length) -> (Batch, Channels, Length)
        if x.dim() == 2:
            x = x.unsqueeze(1) 
            
        # 2. Pass through network
        x = self.layer1(x)
        x = self.res_block(x)
        x = self.layer3(x)
        
        # 3. Pool the spatial/length dimension and flatten
        x = self.global_pool(x)       # Shape becomes (64, 32, 1)
        x = torch.flatten(x, start_dim=1) # Shape becomes (64, 32)
        
        x = self.fc(x)
        return x

In [15]:
def gen_classifier_roc(inputs):
    X_train, X_test, y_train, y_test = train_test_split(inputs, correct.astype(int), test_size=0.2, random_state=123)
    classifier_model = CNNClassifier(X_train.shape[1]).to(device)
    X_train = torch.tensor(X_train).to(device)
    y_train = torch.tensor(y_train).to(torch.long).to(device)
    X_test = torch.tensor(X_test).to(device)
    y_test = torch.tensor(y_test).to(torch.long).to(device)

    optimizer = torch.optim.AdamW(classifier_model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    for _ in tqdm(range(1001)):
        optimizer.zero_grad()
        sample = torch.randperm(X_train.shape[0])[:batch_size]
        pred = classifier_model(X_train[sample])
        loss = torch.nn.functional.cross_entropy(pred, y_train[sample])
        loss.backward()
        optimizer.step()
    classifier_model.eval()
    with torch.no_grad():
        pred = torch.nn.functional.softmax(classifier_model(X_test), dim=1)
        prediction_classes = (pred[:,1]>0.5).type(torch.long).cpu()
        return roc_auc_score(y_test.cpu(), pred[:,1].cpu()), (prediction_classes.numpy()==y_test.cpu().numpy()).mean()

In [16]:
all_results = {}

In [11]:
for results_file in inference_results:
    if results_file not in all_results.keys():
        try:
            del results
        except:
            pass
        try:
            classifier_results = {}
            with open(results_file, "rb") as infile:
                results = pickle.loads(infile.read())
            correct = np.array(results['correct'])

            # logits
            first_logits = np.stack([sp.special.softmax(i[j]) for i, j in zip(results['logits'], results['start_pos'])])
            first_logits_roc, first_logits_acc = gen_classifier_roc(first_logits)
            classifier_results['first_logits_roc'] = first_logits_roc
            classifier_results['first_logits_acc'] = first_logits_acc

            layer = 31
            layer_roc, layer_acc = gen_classifier_roc(np.stack([i[layer] for i in results['first_fully_connected']]))
            classifier_results[f'first_fully_connected_roc_{layer}'] = layer_roc
            classifier_results[f'first_fully_connected_acc_{layer}'] = layer_acc

            # attention
            layer_roc, layer_acc = gen_classifier_roc(np.stack([i[layer] for i in results['first_attention']]))
            classifier_results[f'first_attention_roc_{layer}'] = layer_roc
            classifier_results[f'first_attention_acc_{layer}'] = layer_acc
         
            all_results[results_file] = classifier_results.copy()
        except Exception as err:
            print(err)

100%|██████████| 1001/1001 [00:11<00:00, 87.72it/s]


In [12]:
for k,v in all_results.items():
    for kk, vv in v.items():
        print(kk, vv)

first_logits_roc 0.5738323805414733
first_logits_acc 0.625
first_fully_connected_roc_31 0.6056520332218747
first_fully_connected_acc_31 0.43
first_attention_roc_31 0.6118002372991047
first_attention_acc_31 0.655


### Sequence of multiple attention scores

In [3]:
def gen_classifier_roc(inputs):
    X_train, X_test, y_train, y_test = train_test_split(inputs, correct.astype(int), test_size=0.2, random_state=123)
    classifier_model = FFHallucinationClassifier(X_train.shape[1]).to(device)
    X_train = torch.tensor(X_train).to(device)
    y_train = torch.tensor(y_train).to(torch.long).to(device)
    X_test = torch.tensor(X_test).to(device)
    y_test = torch.tensor(y_test).to(torch.long).to(device)

    optimizer = torch.optim.AdamW(classifier_model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    for _ in tqdm(range(1001)):
        optimizer.zero_grad()
        sample = torch.randperm(X_train.shape[0])[:batch_size]
        pred = classifier_model(X_train[sample])
        loss = torch.nn.functional.cross_entropy(pred, y_train[sample])
        loss.backward()
        optimizer.step()
    classifier_model.eval()
    with torch.no_grad():
        pred = torch.nn.functional.softmax(classifier_model(X_test), dim=1)
        prediction_classes = (pred[:,1]>0.5).type(torch.long).cpu()
        return roc_auc_score(y_test.cpu(), pred[:,1].cpu()), (prediction_classes.numpy()==y_test.cpu().numpy()).mean()

In [5]:
all_results = {}
for results_file in inference_results:
    if results_file not in all_results.keys():
        try:
            del results
        except:
            pass
        try:
            classifier_results = {}
            with open(results_file, "rb") as infile:
                results = pickle.loads(infile.read())
            correct = np.array(results['correct'])

            layers = [20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]
            multi_attn = np.array([[i[layer] for layer in layers] for i in results['first_attention']])
            multi_attn = multi_attn.reshape(multi_attn.shape[0], -1)
            print(f"multi_attn shape: {multi_attn.shape}")
                
            layer_roc, layer_acc = gen_classifier_roc(multi_attn)
            classifier_results[f'first_multi_attention_roc_20-31'] = layer_roc
            classifier_results[f'first_multi_attention_acc_20-31'] = layer_acc

            all_results[results_file] = classifier_results.copy()
        except Exception as err:
            print(err)

multi_attn shape: (1000, 49152)


100%|██████████| 1001/1001 [00:03<00:00, 319.73it/s]


In [6]:
for k,v in all_results.items():
    for kk, vv in v.items():
        print(kk, vv)

first_multi_attention_roc_20-31 0.6705856973357782
first_multi_attention_acc_20-31 0.67


### Treat attention scores as sequence

In [ ]:
import torch
import torch.nn as nn

class RNNHallucinationClassifier(nn.Module):
    def __init__(self, dropout=dropout_gru): 
        super().__init__()
        hidden_dim = 128
        num_layers = 4
        self.gru = nn.GRU(
            input_size=4096, 
            hidden_size=hidden_dim, 
            num_layers=num_layers, 
            dropout=dropout, 
            batch_first=True, 
            bidirectional=False
        )
        self.linear = nn.Linear(hidden_dim, 2)
    
    def forward(self, seq):
        # gru_out shape: [batch_size, num_layers, hidden_dim]
        gru_out, _ = self.gru(seq)
        last_step_out = gru_out[:, -1, :] 
        logits = self.linear(last_step_out)
        
        return logits

In [4]:
with open("results/1000_samples/open_llama_7b_trivia_qa_start-0_end-1000_4_28.pickle", "rb") as infile:
    results = pickle.loads(infile.read())
correct = np.array(results['correct'])
layers = range(20, 32)
multi_attn = np.array([[i[layer] for layer in layers] for i in results['first_attention']])
print(f"multi_attn shape: {multi_attn.shape}")

X_train, X_test, y_train, y_test = train_test_split(multi_attn, correct.astype(int), test_size=0.2, random_state=1234)
X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train, dtype=torch.long).to(device)
X_test = torch.tensor(X_test, dtype=torch.float32).to(device)

rnn_model = RNNHallucinationClassifier().to(device)
optimizer = torch.optim.AdamW(rnn_model.parameters(), lr=learning_rate, weight_decay=weight_decay)

rnn_model.train()
for step in tqdm(range(1001)):
    sample = torch.randperm(X_train.shape[0])[:batch_size]
    optimizer.zero_grad()
    preds = rnn_model(X_train[sample])
    loss = torch.nn.functional.cross_entropy(preds, y_train[sample])
    loss.backward()
    optimizer.step()

rnn_model.eval()
with torch.no_grad():
    pred = torch.nn.functional.softmax(rnn_model(X_test), dim=1)
    prediction_classes = (pred[:,1]>0.5).type(torch.long).cpu().numpy()

multi_attention_roc = roc_auc_score(y_test, pred[:,1].cpu().numpy())
multi_attention_acc = (prediction_classes==y_test).mean()
print(f"Multi-Attention ROC: {multi_attention_roc:.4f}, Accuracy: {multi_attention_acc:.4f}")
            

multi_attn shape: (1000, 12, 4096)


100%|██████████| 1001/1001 [00:07<00:00, 126.45it/s]

Multi-Attention ROC: 0.6217, Accuracy: 0.6550


In [27]:
# with open("results/2500_samples/open_llama_7b_trivia_qa_start-0_end-2500_5_11.pickle", "rb") as infile:
#     results = pickle.loads(infile.read())
correct = np.array(results['correct'])
layers = range(27, 32)
multi_attn = np.array([[i[layer] for layer in layers] for i in results['first_attention']])
print(f"multi_attn shape: {multi_attn.shape}")

X_train, X_test, y_train, y_test = train_test_split(multi_attn, correct.astype(int), test_size=0.2, random_state=1234)
X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train, dtype=torch.long).to(device)
X_test = torch.tensor(X_test, dtype=torch.float32).to(device)

rnn_model = RNNHallucinationClassifier().to(device)
optimizer = torch.optim.AdamW(rnn_model.parameters(), lr=learning_rate, weight_decay=weight_decay)

rnn_model.train()
for step in tqdm(range(1001)):
    sample = torch.randperm(X_train.shape[0])[:batch_size]
    optimizer.zero_grad()
    preds = rnn_model(X_train[sample])
    loss = torch.nn.functional.cross_entropy(preds, y_train[sample])
    loss.backward()
    optimizer.step()

rnn_model.eval()
with torch.no_grad():
    pred = torch.nn.functional.softmax(rnn_model(X_test), dim=1)
    prediction_classes = (pred[:,1]>0.5).type(torch.long).cpu().numpy()

multi_attention_roc = roc_auc_score(y_test, pred[:,1].cpu().numpy())
multi_attention_acc = (prediction_classes==y_test).mean()
print(f"Multi-Attention ROC: {multi_attention_roc:.4f}, Accuracy: {multi_attention_acc:.4f}")
            

multi_attn shape: (2500, 5, 4096)


100%|██████████| 1001/1001 [00:06<00:00, 146.83it/s]

Multi-Attention ROC: 0.6455, Accuracy: 0.6200


In [2]:
from transformers import LlamaTokenizer, LlamaForCausalLM
from pathlib import Path
import torch

model_dir = Path("./.cache/models/")
device = "cuda:1"
model = LlamaForCausalLM.from_pretrained("openlm-research/open_llama_7b",
                                         cache_dir=model_dir,
                                         device_map=device,
                                         # quantization_config=quant_config, 
                                         torch_dtype=torch.bfloat16,
                                         # load_in_4bit=True,
                                         trust_remote_code=True)

tokenizer = LlamaTokenizer.from_pretrained("openlm-research/open_llama_7b", cache_dir=model_dir, trust_remote_code=True)
import torch
from transformers import LlamaTokenizer, LlamaForCausalLM

prompt = 'When was Donald Trump elected as president?'
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)

with torch.no_grad():
    outputs = model(input_ids, output_attentions=True, output_hidden_states=True)
print("Logits shape:", outputs.logits.shape)
print("Number of hidden states:", len(outputs.hidden_states))
print("Hidden state shape:", outputs.hidden_states[0].shape)
print("Number of attention layers:", len(outputs.attentions))
print("Attention shape:", outputs.attentions[0].shape)

The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
LlamaModel is using LlamaSdpaAttention, but `torch.nn.functional.scaled_dot_product_attention` does not support `output_attentions=True`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


Logits shape: torch.Size([1, 9, 32000])
Number of hidden states: 33
Hidden state shape: torch.Size([1, 9, 4096])
Number of attention layers: 32
Attention shape: torch.Size([1, 32, 9, 9])


### Attention maps & contextual embeddings

In [4]:
inference_results = list(Path("./results/10000_samples_llama3.1").rglob("*.pickle"))
print(inference_results)

[PosixPath('results/10000_samples_llama3.1/Llama-3.1-8B-Instruct_trivia_qa_batch_1.pickle'), PosixPath('results/10000_samples_llama3.1/Llama-3.1-8B-Instruct_trivia_qa_batch_0.pickle')]


In [2]:
class FFHallucinationClassifier(torch.nn.Module):
    def __init__(self, input_shape, dropout = dropout_mlp):
        super().__init__()
        self.dropout = dropout
        
        # self.linear_relu_stack = torch.nn.Sequential(
        #     # First hidden layer
        #     torch.nn.Linear(input_shape, 256),
        #     torch.nn.ReLU(),
        #     torch.nn.Dropout(self.dropout),

        #     # Second hidden layer
        #     torch.nn.Linear(256, 256),
        #     torch.nn.ReLU(),
        #     torch.nn.Dropout(self.dropout),

        #     # Third
        #     torch.nn.Linear(256, 128),
        #     torch.nn.ReLU(),
        #     torch.nn.Dropout(self.dropout),

        #     # Fourth
        #     torch.nn.Linear(128, 64),
        #     torch.nn.ReLU(),
        #     torch.nn.Dropout(self.dropout),

        #     torch.nn.Linear(64, 2)
        #     )
        self.linear_relu_stack = torch.nn.Sequential(
            torch.nn.Linear(input_shape, 256),
            torch.nn.ReLU(),
            torch.nn.Dropout(self.dropout),
            torch.nn.Linear(256, 2)
            )

    def forward(self, x):
        logits = self.linear_relu_stack(x)
        return logits
    
class RNNHallucinationClassifier(torch.nn.Module):
    def __init__(self, dropout=dropout_gru):
        super().__init__()
        hidden_dim = 128
        num_layers = 4
        self.gru = torch.nn.GRU(1, hidden_dim, num_layers, dropout=dropout, batch_first=True, bidirectional=False)
        self.linear = torch.nn.Linear(hidden_dim, 2)
    
    def forward(self, seq):
        gru_out, _ = self.gru(seq)
        return self.linear(gru_out)[-1, -1, :]

In [3]:
def gen_classifier_roc(inputs):
    X_train, X_test, y_train, y_test = train_test_split(inputs, correct.astype(int), test_size=0.2, random_state=123)
    classifier_model = FFHallucinationClassifier(X_train.shape[1]).to(device)
    X_train = torch.tensor(X_train).to(device)
    y_train = torch.tensor(y_train).to(torch.long).to(device)
    X_test = torch.tensor(X_test).to(device)
    y_test = torch.tensor(y_test).to(torch.long).to(device)

    optimizer = torch.optim.AdamW(classifier_model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    for _ in tqdm(range(1001)):
        optimizer.zero_grad()
        sample = torch.randperm(X_train.shape[0])[:batch_size]
        pred = classifier_model(X_train[sample])
        loss = torch.nn.functional.cross_entropy(pred, y_train[sample])
        loss.backward()
        optimizer.step()
    classifier_model.eval()
    with torch.no_grad():
        pred = torch.nn.functional.softmax(classifier_model(X_test), dim=1)
        prediction_classes = (pred[:,1]>0.5).type(torch.long).cpu()
        return roc_auc_score(y_test.cpu(), pred[:,1].cpu()), (prediction_classes.numpy()==y_test.cpu().numpy()).mean()

In [6]:
all_results = {}
classifier_results = {}
results = defaultdict(list)
for results_file in tqdm(sorted(inference_results)):
    # if results_file not in all_results.keys():
        # try:
        #     del results
        # except:
        #     pass
        # try:
        #     classifier_results = {}
        #     results = defaultdict(list)
        with open(results_file, "rb") as infile:
            batch_data = pickle.loads(infile.read())
        for key, values in batch_data.items():
            results[key].extend(values)

correct = np.array(results['correct'])
# attributes
X_train, X_test, y_train, y_test = train_test_split(results['attributes_first'], correct.astype(int), test_size=0.2, random_state=1234)
rnn_model = RNNHallucinationClassifier().to(device)
optimizer = torch.optim.AdamW(rnn_model.parameters(), lr=learning_rate, weight_decay=weight_decay)
for step in tqdm(range(1001)):
    x_sub, y_sub = zip(*random.sample(list(zip(X_train, y_train)), batch_size))
    y_sub = torch.tensor(y_sub).to(torch.long).to(device)
    optimizer.zero_grad()
    preds = torch.stack([rnn_model(torch.tensor(i).view(1, -1, 1).to(torch.float).to(device)) for i in x_sub])
    loss = torch.nn.functional.cross_entropy(preds, y_sub)
    loss.backward()
    optimizer.step()

preds = torch.stack([rnn_model(torch.tensor(i).view(1, -1, 1).to(torch.float).to(device)) for i in X_test])
preds = torch.nn.functional.softmax(preds, dim=1)
prediction_classes = (preds[:,1]>0.5).type(torch.long).cpu()
classifier_results['attribution_rnn_roc'] = roc_auc_score(y_test, preds[:,1].detach().cpu().numpy())
classifier_results['attribution_rnn_acc'] = (prediction_classes.numpy()==y_test).mean()

# logits
first_logits = np.stack([sp.special.softmax(i[j]) for i, j in zip(results['logits'], results['start_pos'])])
first_logits_roc, first_logits_acc = gen_classifier_roc(first_logits)
classifier_results['first_logits_roc'] = first_logits_roc
classifier_results['first_logits_acc'] = first_logits_acc

# fully connected
for layer in range(results['first_fully_connected'][0].shape[0]):
    layer_roc, layer_acc = gen_classifier_roc(np.stack([i[layer] for i in results['first_fully_connected']]))
    classifier_results[f'first_fully_connected_roc_{layer}'] = layer_roc
    classifier_results[f'first_fully_connected_acc_{layer}'] = layer_acc

# attention
for layer in range(results['first_attention'][0].shape[0]):
    layer_roc, layer_acc = gen_classifier_roc(np.stack([i[layer] for i in results['first_attention']]))
    classifier_results[f'first_attention_roc_{layer}'] = layer_roc
    classifier_results[f'first_attention_acc_{layer}'] = layer_acc

pooled_embeddings = []
for i in range(len(results['generated_embeddings'])):
    embeddings = results['generated_embeddings'][i]
    # print(f"Embeddings shape for sample {i}: {embeddings}")
    emb_sequence = torch.tensor(embeddings, dtype=torch.float32)
    pooled_emb = torch.mean(emb_sequence, dim=0)
    pooled_embeddings.append(pooled_emb)

pooled_embeddings_tensor = torch.stack(pooled_embeddings)
context_emb_roc, context_emb_acc = gen_classifier_roc(pooled_embeddings_tensor)
classifier_results['context_emb_roc'] = context_emb_roc
classifier_results['context_emb_acc'] = context_emb_acc

        # all_results[results_file] = classifier_results.copy()
        # except Exception as err:
        #     print(err)

100%|██████████| 1001/1001 [00:02<00:00, 363.14it/s]
/tmp/ipykernel_583711/3136129887.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_train = torch.tensor(X_train).to(device)
/tmp/ipykernel_583711/3136129887.py:6: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_test = torch.tensor(X_test).to(device)
100%|██████████| 1001/1001 [00:02<00:00, 416.11it/s]


In [7]:
for k,v in classifier_results.items():
    # for kk, vv in v.items():
    print(k, v)

attribution_rnn_roc 0.5512048443062663
attribution_rnn_acc 0.6985
first_logits_roc 0.6575897586124636
first_logits_acc 0.713
first_fully_connected_roc_0 0.6868488352115062
first_fully_connected_acc_0 0.7235
first_fully_connected_roc_1 0.697361411367426
first_fully_connected_acc_1 0.728
first_fully_connected_roc_2 0.7134423773625487
first_fully_connected_acc_2 0.7375
first_fully_connected_roc_3 0.7228196447354922
first_fully_connected_acc_3 0.7465
first_fully_connected_roc_4 0.7171183798890277
first_fully_connected_acc_4 0.737
first_fully_connected_roc_5 0.7288929596022897
first_fully_connected_acc_5 0.73
first_fully_connected_roc_6 0.7219562661999355
first_fully_connected_acc_6 0.732
first_fully_connected_roc_7 0.7376861949665712
first_fully_connected_acc_7 0.7435
first_fully_connected_roc_8 0.7437873208814001
first_fully_connected_acc_8 0.7465
first_fully_connected_roc_9 0.7620635056011451
first_fully_connected_acc_9 0.7445
first_fully_connected_roc_10 0.7682758757081001
first_fully_c

### Transformer + Residual

In [6]:
import torch
import torch.nn as nn
torch.autograd.set_detect_anomaly(True)

class ResidualBlock(nn.Module):
    def __init__(self, hidden_dim):
        super(ResidualBlock, self).__init__()
        self.fc1 = nn.Linear(hidden_dim, hidden_dim)
        self.relu = nn.ReLU()
        # self.fc2 = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x):
        identity = x
        
        out = self.fc1(x)
        # out = self.relu(out)
        # out = self.fc2(out)
        
        out = out + identity
        out = self.relu(out)
        
        return out

class DNNClassifier(nn.Module):
    def __init__(self, input_size, hidden_size=256, nhead=8, dropout=dropout_mlp):
        super(DNNClassifier, self).__init__()
        
        self.layer1 = nn.Linear(input_size, hidden_size)
        self.layer2 = nn.Linear(hidden_size, hidden_size)
        self.relu = nn.ReLU()
        self.transformer_block = nn.TransformerEncoderLayer(
            d_model=hidden_size,
            nhead=nhead,
            dim_feedforward=hidden_size * 2,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )
        self.res_block = ResidualBlock(hidden_size)
        # self.layer3 = nn.Linear(hidden_size, hidden_size)
        self.layer4 = nn.Linear(hidden_size, 2)
        
    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        
        x = self.layer2(x)
        x = self.relu(x)
        
        x = x.unsqueeze(1)
        x = self.transformer_block(x)
        x = x.squeeze(1)
        
        x = self.res_block(x)
        # x = self.layer3(x)
        # x = self.relu(x)
        x = self.layer4(x)
        
        return x
    
class RNNHallucinationClassifier(torch.nn.Module):
    def __init__(self, dropout=dropout_gru):
        super().__init__()
        hidden_dim = 128
        num_layers = 4
        self.gru = torch.nn.GRU(1, hidden_dim, num_layers, dropout=dropout, batch_first=True, bidirectional=False)
        self.linear = torch.nn.Linear(hidden_dim, 2)
    
    def forward(self, seq):
        gru_out, _ = self.gru(seq)
        return self.linear(gru_out)[-1, -1, :]

In [3]:
def gen_classifier_roc(inputs):
    X_train, X_test, y_train, y_test = train_test_split(inputs, correct.astype(int), test_size=0.2, random_state=123)
    classifier_model = DNNClassifier(X_train.shape[1]).to(device)
    X_train = torch.tensor(X_train).to(device)
    y_train = torch.tensor(y_train).to(torch.long).to(device)
    X_test = torch.tensor(X_test).to(device)
    y_test = torch.tensor(y_test).to(torch.long).to(device)

    optimizer = torch.optim.AdamW(classifier_model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    for _ in tqdm(range(1001)):
        optimizer.zero_grad()
        sample = torch.randperm(X_train.shape[0])[:batch_size]
        pred = classifier_model(X_train[sample])
        loss = torch.nn.functional.cross_entropy(pred, y_train[sample])
        loss.backward()
        optimizer.step()

    classifier_model.eval()
    with torch.no_grad():
        pred = torch.nn.functional.softmax(classifier_model(X_test), dim=1)
        prediction_classes = (pred[:,1]>0.5).type(torch.long).cpu()
        return roc_auc_score(y_test.cpu(), pred[:,1].cpu()), (prediction_classes.numpy()==y_test.cpu().numpy()).mean()

In [4]:
inference_results = list(Path("./results/10000_samples_OpenLlama").rglob("*.pickle"))
print(inference_results)

[PosixPath('results/10000_samples_OpenLlama/open_llama_7b_trivia_qa_start-0_end-10000_6_16.pickle')]


In [7]:
all_results = {}
for results_file in inference_results:
    if results_file not in all_results.keys():
        # try:
        #     del results
        # except:
        #     pass
        try:
            classifier_results = {}
            # with open(results_file, "rb") as infile:
            #     results = pickle.loads(infile.read())
            correct = np.array(results['correct'])
            # infile.close()
    
            # attributes
            X_train, X_test, y_train, y_test = train_test_split(results['attributes_first'], correct.astype(int), test_size=0.2, random_state=1234)
            rnn_model = RNNHallucinationClassifier().to(device)
            optimizer = torch.optim.AdamW(rnn_model.parameters(), lr=learning_rate, weight_decay=weight_decay)
            for step in tqdm(range(1001)):
                x_sub, y_sub = zip(*random.sample(list(zip(X_train, y_train)), batch_size))
                y_sub = torch.tensor(y_sub).to(torch.long).to(device)
                optimizer.zero_grad()
                preds = torch.stack([rnn_model(torch.tensor(i).view(1, -1, 1).to(torch.float).to(device)) for i in x_sub])
                loss = torch.nn.functional.cross_entropy(preds, y_sub)
                loss.backward()
                optimizer.step()
            # torch.save(rnn_model.state_dict(), f"rnn_model.pt")
            
            preds = torch.stack([rnn_model(torch.tensor(i).view(1, -1, 1).to(torch.float).to(device)) for i in X_test])
            preds = torch.nn.functional.softmax(preds, dim=1)
            prediction_classes = (preds[:,1]>0.5).type(torch.long).cpu()
            classifier_results['attribution_rnn_roc'] = roc_auc_score(y_test, preds[:,1].detach().cpu().numpy())
            classifier_results['attribution_rnn_acc'] = (prediction_classes.numpy()==y_test).mean()

            # logits
            first_logits = np.stack([sp.special.softmax(i[j]) for i, j in zip(results['logits'], results['start_pos'])])
            first_logits_roc, first_logits_acc = gen_classifier_roc(first_logits)
            classifier_results['first_logits_roc'] = first_logits_roc
            classifier_results['first_logits_acc'] = first_logits_acc

            # fully connected
            for layer in range(results['first_fully_connected'][0].shape[0]):
                layer_roc, layer_acc = gen_classifier_roc(np.stack([i[layer] for i in results['first_fully_connected']]))
                classifier_results[f'first_fully_connected_roc_{layer}'] = layer_roc
                classifier_results[f'first_fully_connected_acc_{layer}'] = layer_acc

            # attention
            for layer in range(results['first_attention'][0].shape[0]):
                layer_roc, layer_acc = gen_classifier_roc(np.stack([i[layer] for i in results['first_attention']]))
                classifier_results[f'first_attention_roc_{layer}'] = layer_roc
                classifier_results[f'first_attention_acc_{layer}'] = layer_acc
            
            pooled_embeddings = []
            for i in range(len(results['generated_embeddings'])):
                embeddings = results['generated_embeddings'][i]
                # print(f"Embeddings shape for sample {i}: {embeddings}")
                emb_sequence = torch.tensor(embeddings, dtype=torch.float32)
                pooled_emb = torch.mean(emb_sequence, dim=0)
                pooled_embeddings.append(pooled_emb)

            pooled_embeddings_tensor = torch.stack(pooled_embeddings)
            context_emb_roc, context_emb_acc = gen_classifier_roc(pooled_embeddings_tensor)
            classifier_results['context_emb_roc'] = context_emb_roc
            classifier_results['context_emb_acc'] = context_emb_acc

            all_results[results_file] = classifier_results.copy()
        except Exception as err:
            print(err)

100%|██████████| 1001/1001 [00:28<00:00, 34.92it/s]
/tmp/ipykernel_941884/331616126.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_train = torch.tensor(X_train).to(device)
/tmp/ipykernel_941884/331616126.py:6: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_test = torch.tensor(X_test).to(device)
100%|██████████| 1001/1001 [00:29<00:00, 34.39it/s]


In [9]:
for k,v in classifier_results.items():
    # for kk, vv in v.items():
        print(k, v)

attribution_rnn_roc 0.6227252012924183
attribution_rnn_acc 0.635
first_logits_roc 0.6959075320291908
first_logits_acc 0.6585
first_fully_connected_roc_0 0.6727456337122114
first_fully_connected_acc_0 0.65
first_fully_connected_roc_1 0.6556449490582352
first_fully_connected_acc_1 0.655
first_fully_connected_roc_2 0.6876438546144173
first_fully_connected_acc_2 0.666
first_fully_connected_roc_3 0.6877691726088282
first_fully_connected_acc_3 0.658
first_fully_connected_roc_4 0.7000210951957259
first_fully_connected_acc_4 0.6715
first_fully_connected_roc_5 0.7031488234728958
first_fully_connected_acc_5 0.673
first_fully_connected_roc_6 0.7103191013864347
first_fully_connected_acc_6 0.675
first_fully_connected_roc_7 0.7146321290274069
first_fully_connected_acc_7 0.6825
first_fully_connected_roc_8 0.7120338692766228
first_fully_connected_acc_8 0.675
first_fully_connected_roc_9 0.7082001829642719
first_fully_connected_acc_9 0.676
first_fully_connected_roc_10 0.7123085245477065
first_fully_conn